# Hebbian Learning Tutorial with PyTorch and CIFAR-10

This tutorial demonstrates how to implement Hebbian learning using PyTorch, the CIFAR-10 dataset, and a small CNN model.

## What is Hebbian Learning?

Hebbian learning is based on the principle: "Neurons that fire together, wire together." This means that when two neurons are activated simultaneously, the connection between them is strengthened.

In this tutorial, we'll implement a Hebbian learning approach that:
1. Uses a reward function (1 for correct cat classification, 0 otherwise)
2. Maintains trace matrices for model parameters
3. Uses a custom optimizer that combines rewards and traces to update weights

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load and Prepare CIFAR-10 Dataset

CIFAR-10 contains 60,000 32x32 color images in 10 classes. We'll focus on detecting cats (class 3).

In [ ]:
# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

# Use a subset for faster training in this tutorial
train_subset = Subset(trainset, range(5000))
test_subset = Subset(testset, range(1000))

trainloader = DataLoader(train_subset, batch_size=32, shuffle=True)
testloader = DataLoader(test_subset, batch_size=32, shuffle=False)

# CIFAR-10 classes
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
CAT_CLASS = 3

print(f'Training samples: {len(train_subset)}')
print(f'Test samples: {len(test_subset)}')
print(f'Cat class index: {CAT_CLASS}')

## Define a Small CNN Model

We'll create a simple convolutional neural network for image classification.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super(SmallCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)
        
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        
        # Flatten
        x = x.view(-1, 32 * 8 * 8)
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Initialize model
model = SmallCNN().to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters())}')

## Reward Function

The reward function returns 1 if the model correctly classifies an image as a cat, and 0 otherwise.
This is the key component that guides Hebbian learning.

In [ ]:
def reward_function(predictions, labels):
    """
    Returns 1 for each correct cat classification, 0 otherwise.
    
    Args:
        predictions: Model output logits (batch_size, num_classes)
        labels: True labels (batch_size,)
    
    Returns:
        rewards: Tensor of shape (batch_size,) with 1s and 0s
    """
    # Get predicted classes
    _, predicted = torch.max(predictions, 1)
    
    # Reward = 1 if prediction is correct AND it's a cat
    # Reward = 0 otherwise
    correct_predictions = (predicted == labels)
    is_cat = (labels == CAT_CLASS)
    
    rewards = (correct_predictions & is_cat).float()
    
    return rewards

# Test the reward function
test_predictions = torch.tensor([[0.1, 0.2, 0.3, 0.9, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                                  [0.9, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]])
test_labels = torch.tensor([3, 3])  # Both are cats
test_rewards = reward_function(test_predictions, test_labels)
print(f'Test rewards: {test_rewards}')  # Should be [1.0, 0.0]

## Trace Matrix Implementation

Trace matrices store information about recent parameter activity. They help implement the Hebbian principle by tracking which parameters were active during successful predictions.

In [ ]:
class TraceMatrix:
    """
    Maintains eligibility traces for model parameters.
    Traces decay over time and are updated based on parameter gradients.
    """
    def __init__(self, model, decay=0.9):
        """
        Args:
            model: PyTorch model
            decay: Decay factor for traces (lambda in RL terminology)
        """
        self.decay = decay
        self.traces = {}
        
        # Initialize traces for all parameters
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.traces[name] = torch.zeros_like(param.data)
    
    def update(self, model):
        """
        Update traces based on current gradients.
        Trace = decay * trace + gradient
        """
        for name, param in model.named_parameters():
            if param.requires_grad and param.grad is not None:
                # Update trace: decay old trace and add new gradient
                self.traces[name] = self.decay * self.traces[name] + param.grad.data
    
    def reset(self):
        """Reset all traces to zero."""
        for name in self.traces:
            self.traces[name].zero_()
    
    def get_trace(self, name):
        """Get trace for a specific parameter."""
        return self.traces.get(name, None)

# Initialize trace matrix
trace_matrix = TraceMatrix(model, decay=0.9)
print(f'Initialized traces for {len(trace_matrix.traces)} parameter groups')

## Custom Hebbian Optimizer

This optimizer combines the reward signal with trace values to update model parameters according to Hebbian learning principles.

In [ ]:
class HebbianOptimizer:
    """
    Custom optimizer that implements Hebbian learning.
    Updates parameters based on: reward * trace * learning_rate
    """
    def __init__(self, model, trace_matrix, learning_rate=0.001):
        """
        Args:
            model: PyTorch model
            trace_matrix: TraceMatrix instance
            learning_rate: Learning rate for parameter updates
        """
        self.model = model
        self.trace_matrix = trace_matrix
        self.learning_rate = learning_rate
    
    def step(self, reward):
        """
        Update model parameters using Hebbian rule.
        
        Args:
            reward: Scalar reward value (averaged over batch)
        """
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if param.requires_grad:
                    trace = self.trace_matrix.get_trace(name)
                    if trace is not None:
                        # Hebbian update: increase weights where trace and reward align
                        # The trace captures what was active, reward signals if it was good
                        param.data += self.learning_rate * reward * trace
    
    def zero_grad(self):
        """Zero out gradients in the model."""
        self.model.zero_grad()

# Initialize optimizer
optimizer = HebbianOptimizer(model, trace_matrix, learning_rate=0.0001)
print('Hebbian optimizer initialized')

## Training Loop with Hebbian Learning

Now we'll train the model using our Hebbian learning approach.

In [ ]:
def train_hebbian(model, trainloader, optimizer, trace_matrix, num_epochs=5):
    """
    Train model using Hebbian learning.
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    
    history = {
        'loss': [],
        'reward': [],
        'accuracy': []
    }
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        running_reward = 0.0
        correct = 0
        total = 0
        
        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            
            # Calculate loss (for monitoring and gradient computation)
            loss = criterion(outputs, labels)
            
            # Backward pass to compute gradients
            loss.backward()
            
            # Update traces with current gradients
            trace_matrix.update(model)
            
            # Calculate rewards
            rewards = reward_function(outputs, labels)
            avg_reward = rewards.mean().item()
            
            # Hebbian update using average reward
            optimizer.step(avg_reward)
            
            # Statistics
            running_loss += loss.item()
            running_reward += avg_reward
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (i + 1) % 50 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(trainloader)}], '
                      f'Loss: {running_loss/50:.4f}, Reward: {running_reward/50:.4f}')
                running_loss = 0.0
                running_reward = 0.0
        
        # Epoch statistics
        epoch_acc = 100 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}] - Accuracy: {epoch_acc:.2f}%\n')
        
        history['accuracy'].append(epoch_acc)
    
    return history

# Train the model
print('Starting Hebbian learning training...\n')
history = train_hebbian(model, trainloader, optimizer, trace_matrix, num_epochs=3)

## Evaluation

Let's evaluate the model's performance, especially on cat classification.

In [ ]:
def evaluate_model(model, testloader):
    """
    Evaluate model on test set.
    """
    model.eval()
    
    correct = 0
    total = 0
    cat_correct = 0
    cat_total = 0
    
    class_correct = [0] * 10
    class_total = [0] * 10
    
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Cat-specific accuracy
            cat_mask = (labels == CAT_CLASS)
            cat_total += cat_mask.sum().item()
            cat_correct += ((predicted == labels) & cat_mask).sum().item()
            
            # Per-class accuracy
            for i in range(10):
                class_mask = (labels == i)
                class_total[i] += class_mask.sum().item()
                class_correct[i] += ((predicted == labels) & class_mask).sum().item()
    
    overall_acc = 100 * correct / total
    print(f'Overall Test Accuracy: {overall_acc:.2f}%')
    
    if cat_total > 0:
        cat_acc = 100 * cat_correct / cat_total
        print(f'Cat Classification Accuracy: {cat_acc:.2f}% ({cat_correct}/{cat_total})')
    
    print('\nPer-class Accuracy:')
    for i in range(10):
        if class_total[i] > 0:
            acc = 100 * class_correct[i] / class_total[i]
            print(f'  {classes[i]:>8s}: {acc:5.2f}%')
    
    return overall_acc, cat_acc if cat_total > 0 else 0

# Evaluate the model
print('\nEvaluating model...\n')
overall_acc, cat_acc = evaluate_model(model, testloader)

## Visualization

Let's visualize some predictions, especially focusing on cat images.

In [ ]:
def imshow(img):
    """Display an image."""
    img = img / 2 + 0.5  # Unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

def visualize_predictions(model, testloader, num_images=8):
    """
    Visualize model predictions on test images.
    """
    model.eval()
    
    # Get a batch of test images
    dataiter = iter(testloader)
    images, labels = next(dataiter)
    images, labels = images.to(device), labels.to(device)
    
    # Get predictions
    with torch.no_grad():
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
    
    # Move back to CPU for visualization
    images = images.cpu()
    labels = labels.cpu()
    predicted = predicted.cpu()
    
    # Plot images
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.ravel()
    
    for i in range(min(num_images, len(images))):
        ax = axes[i]
        img = images[i]
        img = img / 2 + 0.5  # Unnormalize
        npimg = img.numpy()
        ax.imshow(np.transpose(npimg, (1, 2, 0)))
        
        true_label = classes[labels[i]]
        pred_label = classes[predicted[i]]
        
        # Color code: green if correct, red if wrong
        color = 'green' if labels[i] == predicted[i] else 'red'
        
        # Highlight cat predictions
        title = f'True: {true_label}\nPred: {pred_label}'
        if labels[i] == CAT_CLASS:
            title = '🐱 ' + title
        
        ax.set_title(title, color=color, fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize predictions
visualize_predictions(model, testloader)

## Summary

In this tutorial, we've implemented a Hebbian learning system with the following components:

1. **CNN Model**: A small convolutional neural network for image classification
2. **Reward Function**: Returns 1 for correct cat classifications, 0 otherwise
3. **Trace Matrix**: Maintains eligibility traces of parameter activity
4. **Hebbian Optimizer**: Updates parameters based on `reward × trace × learning_rate`

### Key Insights:

- **Hebbian Principle**: The model strengthens connections (parameters) that were active during rewarded outcomes
- **Trace Matrix**: Acts as a memory of recent parameter activity, implementing eligibility traces
- **Reward Signal**: Guides learning by indicating which predictions were successful
- **Selective Learning**: The model focuses on learning from cat classifications specifically

### Potential Improvements:

- Experiment with different trace decay rates
- Try different learning rates
- Extend to multi-class rewards
- Implement more sophisticated reward functions
- Compare with standard backpropagation training

This approach demonstrates how classical Hebbian learning principles can be implemented in modern deep learning frameworks!